In [24]:
from google.colab import files
files.upload()

Saving sap-order-to-cash-dataset.zip to sap-order-to-cash-dataset (1).zip


{'sap-order-to-cash-dataset (1).zip': b'PK\x03\x04\n\x00\x00\x00\x00\x00\x08\x99n\\\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\r\x00\x1c\x00sap-o2c-data/UT\t\x00\x03\xc8d\xb5i\xd3p\xb5iux\x0b\x00\x01\x04\xe8\x03\x00\x00\x04\xe8\x03\x00\x00PK\x03\x04\n\x00\x00\x00\x00\x00\x08\x99n\\\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00*\x00\x1c\x00sap-o2c-data/customer_company_assignments/UT\t\x00\x03\xc7d\xb5i\xc8d\xb5iux\x0b\x00\x01\x04\xe8\x03\x00\x00\x04\xe8\x03\x00\x00PK\x03\x04\x14\x00\x00\x00\x08\x00\x08\x99n\\&}\x1a\xa5\x10\x01\x00\x00\xda\x0b\x00\x00G\x00\x1c\x00sap-o2c-data/customer_company_assignments/part-20251119-133436-59.jsonlUT\t\x00\x03\xc7d\xb5i\xc8d\xb5iux\x0b\x00\x01\x04\xe8\x03\x00\x00\x04\xe8\x03\x00\x00\xed\xd5AK\xc30\x14\x07\xf0\xbb\x9fb\xe4\xbcC\xd3Q\x9c\xde\xba\x8ac\xa02\xc4\x8b\xdeb\xf2ta\xe9{%I\xc52\xfc\xee6n\x8a\x94\x96\xdd\xd6\xc3\xd2c\xfe\xf9\xbf\x84\x1f\x81\xee\x98\xac\x9d\xa7\x12,\xbb\x9e\xb0\x19O\xda\x8f\'s6\x9d0Ie%\xb0)HA\xc8\xf2Eq\x13\x96\x85\x94T\xa3\

In [25]:
import zipfile

with zipfile.ZipFile("sap-order-to-cash-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

In [26]:
import os

for root, dirs, files in os.walk("data"):
    print("Folder:", root)
    print("Files:", files)
    print("-"*40)

Folder: data
Files: []
----------------------------------------
Folder: data/sap-o2c-data
Files: []
----------------------------------------
Folder: data/sap-o2c-data/customer_sales_area_assignments
Files: ['part-20251119-133437-884.jsonl']
----------------------------------------
Folder: data/sap-o2c-data/product_plants
Files: ['part-20251119-133440-467.jsonl', 'part-20251119-133439-488.jsonl', 'part-20251119-133439-232.jsonl', 'part-20251119-133439-814.jsonl']
----------------------------------------
Folder: data/sap-o2c-data/sales_order_schedule_lines
Files: ['part-20251119-133430-302.jsonl', 'part-20251119-133430-979.jsonl']
----------------------------------------
Folder: data/sap-o2c-data/sales_order_items
Files: ['part-20251119-133430-214.jsonl', 'part-20251119-133429-452.jsonl']
----------------------------------------
Folder: data/sap-o2c-data/business_partners
Files: ['part-20251119-133435-168.jsonl']
----------------------------------------
Folder: data/sap-o2c-data/product_

In [27]:
import pandas as pd
import os

def load_jsonl_data(base_path):
    tables = {}

    for root, dirs, files in os.walk(base_path):
        folder_name = os.path.basename(root)

        jsonl_files = [f for f in files if f.endswith(".jsonl")]

        if jsonl_files:
            df_list = []

            for file in jsonl_files:
                path = os.path.join(root, file)

                df = pd.read_json(path, lines=True)
                df_list.append(df)

            combined_df = pd.concat(df_list, ignore_index=True)

            tables[folder_name] = combined_df

    return tables


tables = load_jsonl_data("data/sap-o2c-data")

In [28]:
for name, df in tables.items():
    print(name, df.shape)

customer_sales_area_assignments (28, 19)
product_plants (3036, 9)
sales_order_schedule_lines (179, 6)
sales_order_items (167, 13)
business_partners (8, 19)
product_storage_locations (16723, 5)
outbound_delivery_headers (86, 13)
payments_accounts_receivable (120, 23)
billing_document_cancellations (80, 14)
customer_company_assignments (8, 13)
plants (44, 14)
journal_entry_items_accounts_receivable (123, 22)
billing_document_items (245, 9)
business_partner_addresses (8, 20)
products (69, 17)
product_descriptions (69, 3)
billing_document_headers (163, 14)
outbound_delivery_items (137, 11)
sales_order_headers (100, 24)


In [29]:
!pip install openai pandas networkx

In [30]:
!pip install openai pandas networkx sqlite3

ERROR: Could not find a version that satisfies the requirement sqlite3 (from versions: none)
ERROR: No matching distribution found for sqlite3


In [31]:
import json

def clean_dataframe(df):
    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, dict)).any():
            df[col] = df[col].apply(lambda x: json.dumps(x) if isinstance(x, dict) else x)

        if df[col].apply(lambda x: isinstance(x, list)).any():
            df[col] = df[col].apply(lambda x: json.dumps(x) if isinstance(x, list) else x)

    return df

In [32]:
for name, df in tables.items():
    tables[name] = clean_dataframe(df)

In [33]:
import sqlite3

conn = sqlite3.connect("data.db")

for name, df in tables.items():
    df.to_sql(name, conn, if_exists="replace", index=False)

print("DB READY 💀🔥")

DB READY 💀🔥


In [43]:
# =========================
# INSTALL
# =========================
!pip install openai pandas networkx

# =========================
# IMPORTS
# =========================
import pandas as pd
import os
import json
import sqlite3
from openai import OpenAI

# =========================
# LOAD JSONL DATA
# =========================
def load_jsonl_data(base_path):
    tables = {}

    for root, dirs, files in os.walk(base_path):
        folder_name = os.path.basename(root)

        jsonl_files = []
        for f in files:
            if f.endswith(".jsonl"):
                jsonl_files.append(f)

        if len(jsonl_files) > 0:
            df_list = []

            for file in jsonl_files:
                path = os.path.join(root, file)
                df = pd.read_json(path, lines=True)
                df_list.append(df)

            combined_df = pd.concat(df_list, ignore_index=True)
            tables[folder_name] = combined_df

    return tables


tables = load_jsonl_data("data/sap-o2c-data")
print("DATA LOADED:", list(tables.keys()))

# =========================
# CLEAN DATA
# =========================
def clean_dataframe(df):
    for col in df.columns:
        df[col] = df[col].apply(
            lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
        )
    return df

for name in tables:
    tables[name] = clean_dataframe(tables[name])

print("DATA CLEANED")

# =========================
# DATABASE
# =========================
conn = sqlite3.connect("data.db")

for name, df in tables.items():
    df.to_sql(name, conn, if_exists="replace", index=False)

print("DB READY 💀")



DATA LOADED: ['customer_sales_area_assignments', 'product_plants', 'sales_order_schedule_lines', 'sales_order_items', 'business_partners', 'product_storage_locations', 'outbound_delivery_headers', 'payments_accounts_receivable', 'billing_document_cancellations', 'customer_company_assignments', 'plants', 'journal_entry_items_accounts_receivable', 'billing_document_items', 'business_partner_addresses', 'products', 'product_descriptions', 'billing_document_headers', 'outbound_delivery_items', 'sales_order_headers']
DATA CLEANED
DB READY 💀


In [45]:
# =========================
# LLM SETUP
# =========================
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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"
)


In [48]:
# =========================
# 🔥 SCHEMA
# =========================
def get_schema():
    schema = ""
    for name, df in tables.items():
        schema += name + "(" + ", ".join(df.columns) + ")\n"
    return schema


# =========================
# 🔥 SQL CLEANER
# =========================
import re

def clean_sql(sql_text):

    match = re.search(r"```sql(.*?)```", sql_text, re.DOTALL)
    if match:
        return match.group(1).strip()

    match = re.search(r"(SELECT .*?;)", sql_text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()

    lines = sql_text.split("\n")
    clean_lines = []

    for line in lines:
        line = line.strip()
        if line.startswith("#") or line.startswith("--") or line == "":
            continue
        clean_lines.append(line)

    return " ".join(clean_lines).strip()


# =========================
# 🔥 SQL GENERATOR
# =========================
def generate_sql(query):

    prompt = f"""
You are a STRICT SQLite SQL generator.

RULES:
- ONLY return SQL
- NO explanation
- NO markdown
- NO comments

Schema:
{get_schema()}

Query: {query}
"""

    res = client.chat.completions.create(
        model="deepseek/deepseek-chat",
        messages=[
            {"role": "system", "content": "You only generate SQL."},
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=300
    )

    raw_sql = res.choices[0].message.content.strip()

    return clean_sql(raw_sql)


# =========================
# 🔥 RUN SQL
# =========================
def run_sql(sql):
    try:
        return pd.read_sql_query(sql, conn)
    except Exception as e:
        return f"SQL ERROR: {str(e)}"


# =========================
# 🔥 FINAL SYSTEM
# =========================
def ask_system(query):

    if any(x in query.lower() for x in ["weather", "poem", "who is", "joke"]):
        return {"answer": "This system only supports dataset queries."}

    sql = generate_sql(query)
    print("🔥 SQL:", sql)

    result = run_sql(sql)
    print("📊 RESULT:\n", result)

    return {
        "query": query,
        "sql": sql,
        "result": str(result)
    }


# =========================
# 🔥 TEST
# =========================
ask_system("show all orders")
ask_system("list all products")
ask_system("find orders without invoices")

🔥 SQL: SELECT * FROM sales_order_headers;
📊 RESULT:
     salesOrder salesOrderType salesOrganization  distributionChannel  \
0       740506             OR              ABCD                    5   
1       740507             OR              ABCD                    5   
2       740508             OR              ABCD                    5   
3       740509             OR              ABCD                   11   
4       740510             OR              ABCD                   11   
..         ...            ...               ...                  ...   
95      740601             OR              ABCD                   11   
96      740602             OR              ABCD                   11   
97      740603             OR              ABCD                   11   
98      740604             OR              ABCD                   11   
99      740605             OR              ABCD                   11   

    organizationDivision salesGroup salesOffice  soldToParty  \
0                 

{'query': 'find orders without invoices',
 'sql': 'SELECT so.salesOrder \nFROM sales_order_headers so\nLEFT JOIN billing_document_headers bdh ON so.salesOrder = bdh.salesDocument\nWHERE bdh.salesDocument IS NULL;',
 'result': "SQL ERROR: Execution failed on sql 'SELECT so.salesOrder \nFROM sales_order_headers so\nLEFT JOIN billing_document_headers bdh ON so.salesOrder = bdh.salesDocument\nWHERE bdh.salesDocument IS NULL;': no such column: bdh.salesDocument"}

In [55]:
def generate_sql(query):

    q = query.lower()

    # 💀 RULE-BASED FIX (VERY IMPORTANT)
    if "order" in q:
        return "SELECT * FROM sales_order_headers LIMIT 10"

    if "product" in q:
        return "SELECT * FROM products LIMIT 10"

    if "customer" in q:
        return "SELECT * FROM business_partners LIMIT 10"

    if "orders without invoices" in q:
        return """
        SELECT so.sales_order
        FROM sales_order_headers so
        LEFT JOIN outbound_delivery_headers od
        ON so.sales_order = od.sales_order
        LEFT JOIN billing_document_headers bd
        ON od.delivery_document = bd.delivery_document
        WHERE bd.billing_document IS NULL
        """

    # 🔥 LLM fallback (rare case)
    prompt = f"""
You are a STRICT SQLite SQL generator.

ONLY return SQL.

Schema:
{get_schema()}

Query: {query}
"""

    res = client.chat.completions.create(
        model="deepseek/deepseek-chat",
        messages=[
            {"role": "system", "content": "You only generate SQL."},
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=200
    )

    raw_sql = res.choices[0].message.content.strip()

    return clean_sql(raw_sql)

In [56]:
def run_sql(sql):
    try:
        if not sql.strip().endswith(";"):
            sql += ";"
        return pd.read_sql_query(sql, conn)
    except Exception as e:
        return f"SQL ERROR: {str(e)}"

In [57]:
ask_system("show all orders")
ask_system("list all products")
ask_system("find orders without invoices")

🔥 SQL: SELECT * FROM sales_order_headers LIMIT 10
📊 RESULT:
    salesOrder salesOrderType salesOrganization  distributionChannel  \
0      740506             OR              ABCD                    5   
1      740507             OR              ABCD                    5   
2      740508             OR              ABCD                    5   
3      740509             OR              ABCD                   11   
4      740510             OR              ABCD                   11   
5      740511             OR              ABCD                   11   
6      740512             OR              ABCD                   11   
7      740513             OR              ABCD                   11   
8      740514             OR              ABCD                   11   
9      740515             OR              ABCD                   11   

   organizationDivision salesGroup salesOffice  soldToParty  \
0                    99                           310000108   
1                    99        

{'query': 'find orders without invoices',
 'sql': 'SELECT * FROM sales_order_headers LIMIT 10',
 'result': '   salesOrder salesOrderType salesOrganization  distributionChannel  \\\n0      740506             OR              ABCD                    5   \n1      740507             OR              ABCD                    5   \n2      740508             OR              ABCD                    5   \n3      740509             OR              ABCD                   11   \n4      740510             OR              ABCD                   11   \n5      740511             OR              ABCD                   11   \n6      740512             OR              ABCD                   11   \n7      740513             OR              ABCD                   11   \n8      740514             OR              ABCD                   11   \n9      740515             OR              ABCD                   11   \n\n   organizationDivision salesGroup salesOffice  soldToParty  \\\n0                    99         

In [73]:
def generate_sql(query):

    q = query.lower()

    # 🔥 ORDERS WITHOUT INVOICES
    if "orders without invoices" in q or "not billed" in q:
        return """
        SELECT DISTINCT so.salesOrder
        FROM sales_order_headers so
        LEFT JOIN outbound_delivery_items odi
            ON so.salesOrder = odi.referenceSDDocument
        LEFT JOIN billing_document_items bdi
            ON odi.deliveryDocument = bdi.referenceSDDocument
        WHERE bdi.billingDocument IS NULL
        """

    # 🔥 TRACE FLOW
    if "trace" in q or "flow" in q:
        return """
        SELECT
            so.salesOrder,
            odi.deliveryDocument,
            bdi.billingDocument
        FROM sales_order_headers so
        LEFT JOIN outbound_delivery_items odi
            ON so.salesOrder = odi.referenceSDDocument
        LEFT JOIN billing_document_items bdi
            ON odi.deliveryDocument = bdi.referenceSDDocument
        """

    # 🔥 HIGHEST BILLING
    if "highest billing" in q:
        return """
        SELECT material, COUNT(*) as total_bills
        FROM billing_document_items
        GROUP BY material
        ORDER BY total_bills DESC
        LIMIT 5
        """

    # 🔥 BASIC QUERIES
    if "product" in q:
        return "SELECT * FROM products LIMIT 10"

    if "customer" in q:
        return "SELECT * FROM business_partners LIMIT 10"

    if "order" in q:
        return "SELECT * FROM sales_order_headers LIMIT 10"

    if "delivery" in q:
        return "SELECT * FROM outbound_delivery_headers LIMIT 10"

    if "invoice" in q or "billing" in q:
        return "SELECT * FROM billing_document_headers LIMIT 10"

    # 🔥 LLM FALLBACK
    prompt = f"""
Generate ONLY SQL.

Schema:
{get_schema()}

Query: {query}
"""

    res = client.chat.completions.create(
        model="deepseek/deepseek-chat",
        messages=[
            {"role": "system", "content": "Only SQL."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    raw_sql = res.choices[0].message.content.strip()

    return clean_sql(raw_sql)

In [74]:
ask_system("show all orders")
ask_system("list all products")
ask_system("find orders without invoices")
ask_system("which products have highest billing documents")

🔥 SQL: SELECT * FROM sales_order_headers LIMIT 10
📊 RESULT:
    salesOrder salesOrderType salesOrganization  distributionChannel  \
0      740506             OR              ABCD                    5   
1      740507             OR              ABCD                    5   
2      740508             OR              ABCD                    5   
3      740509             OR              ABCD                   11   
4      740510             OR              ABCD                   11   
5      740511             OR              ABCD                   11   
6      740512             OR              ABCD                   11   
7      740513             OR              ABCD                   11   
8      740514             OR              ABCD                   11   
9      740515             OR              ABCD                   11   

   organizationDivision salesGroup salesOffice  soldToParty  \
0                    99                           310000108   
1                    99        

{'query': 'which products have highest billing documents',
 'sql': '\n        SELECT material, COUNT(*) as total_bills\n        FROM billing_document_items\n        GROUP BY material\n        ORDER BY total_bills DESC\n        LIMIT 5\n        ',
 'result': '         material  total_bills\n0  S8907367039280           22\n1  S8907367008620           22\n2  S8907367042006           16\n3  S8907367013532           15\n4  S8907367001003            9'}

In [75]:
from google.colab import files
files.download("data.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>